In [27]:
import duckdb
print(duckdb.__version__)

1.5.5


In [28]:
import duckdb

path = "~/projects/statsbomb-data/data/events/3754217.json"
duckdb.sql(f"SELECT * FROM read_json_auto('{path}') LIMIT 5").show()
print()
import os
path = os.path.expanduser("~/projects/statsbomb-data/data/events/3754217.json")
duckdb.sql(f"SELECT * FROM read_json_auto('{path}') LIMIT 5").show()

┌──────────────────────────────────────┬───────┬────────┬──────────────┬────────┬────────┬───────────────────────────────────┬────────────┬───────────────────────────────────┬───────────────────────────────────┬───────────────────────────────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [29]:
query = f"""
SELECT type.name AS event_type, COUNT(*) AS n
FROM read_json_auto('{path}')
GROUP BY type.name
ORDER BY n DESC
"""
duckdb.sql(query).show()

┌───────────────────┬───────┐
│    event_type     │   n   │
│      varchar      │ int64 │
├───────────────────┼───────┤
│ Pass              │  1046 │
│ Ball Receipt*     │   963 │
│ Carry             │   814 │
│ Pressure          │   320 │
│ Ball Recovery     │   106 │
│ Duel              │    78 │
│ Dribble           │    52 │
│ Clearance         │    42 │
│ Block             │    41 │
│ Goal Keeper       │    34 │
│   ·               │     · │
│   ·               │     · │
│   ·               │     · │
│ Shield            │     3 │
│ Bad Behaviour     │     3 │
│ Tactical Shift    │     3 │
│ Referee Ball-Drop │     2 │
│ Starting XI       │     2 │
│ Player On         │     1 │
│ Player Off        │     1 │
│ Own Goal Against  │     1 │
│ Own Goal For      │     1 │
│ Error             │     1 │
└───────────────────┴───────┘
  31 rows         2 columns
  (20 shown)                



In [30]:
query = f"""
SELECT
    3754217 AS match_id,
    id,
    index,
    minute,
    second,
    timestamp,
    type.name AS type,
    possession,
    play_pattern.name AS play_pattern,
    team.name AS team,
    player.name AS player,
    position.name AS position,
    location[1] AS x,
    location[2] AS y,
    duration,
    under_pressure
FROM read_json_auto('{path}')
ORDER BY index
"""
duckdb.sql(query).show()

┌──────────┬──────────────────────────────────────┬───────┬────────┬────────┬──────────────┬───────────────┬────────────┬────────────────┬─────────┬─────────────────────────────────┬───────────────────────────┬────────┬────────┬──────────┬────────────────┐
│ match_id │                  id                  │ index │ minute │ second │  timestamp   │     type      │ possession │  play_pattern  │  team   │             player              │         position          │   x    │   y    │ duration │ under_pressure │
│  int32   │                 uuid                 │ int64 │ int64  │ int64  │     time     │    varchar    │   int64    │    varchar     │ varchar │             varchar             │          varchar          │ double │ double │  double  │    boolean     │
├──────────┼──────────────────────────────────────┼───────┼────────┼────────┼──────────────┼───────────────┼────────────┼────────────────┼─────────┼─────────────────────────────────┼───────────────────────────┼────────┼────────┼─

In [31]:
import os

output_path = os.path.expanduser("~/projects/rolefit/data/processed/match_3754217_events.parquet")
os.makedirs(os.path.dirname(output_path), exist_ok=True)

duckdb.sql(query).write_parquet(output_path)